# Reflectance Exercise 7 — measuring an operational spatial footprint

Plot registered scans across black rectangles, estimate a one-dimensional effective footprint, and predict positions reserved before analysis.

Start with the supplied synthetic example so that every cell runs before you have collected data. The example demonstrates the analysis route; it is not evidence about your robot and its values are not coursework answers. You do not need to understand or edit the example-generation cell.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
rng = np.random.default_rng(2026)


## 1. Use the example or your own data

Leave `USE_EXAMPLE_DATA` set to `True` for your first run. To use your measurements, upload the CSV, change it to `False`, and enter its filename. This is the main data-loading setting you need to edit.

Expected raw CSV columns: the seven Exercise 3 fields plus `rectangle_width_mm`, `direction`, and `position_mm`.

Retain `sensor_num` and `sensor_reading`; geometry fields describe the independently measured target and placement.


In [ ]:
USE_EXAMPLE_DATA = True
CSV_FILENAME = "reflectance_exercise07.csv"

print("Using:", "synthetic example" if USE_EXAMPLE_DATA else CSV_FILENAME)


## 2. Create the small synthetic example

The example simulates two traverse directions for one sensor. Its five-millimetre footprint is illustrative only.


In [ ]:
rows = []
positions_mm = np.arange(-8.0, 8.01, 0.5)
for trial_num, (direction, ordered_positions) in enumerate([
    ("left_to_right", positions_mm), ("right_to_left", positions_mm[::-1])
], 1):
    for rectangle_width_mm in [2.0, 4.0, 6.0, 10.0]:
        for sample_num, position_mm in enumerate(ordered_positions, 1):
            rectangle_left, rectangle_right = -rectangle_width_mm / 2, rectangle_width_mm / 2
            footprint_left, footprint_right = position_mm - 2.5, position_mm + 2.5
            overlap_mm = max(0.0, min(rectangle_right, footprint_right) - max(rectangle_left, footprint_left))
            dark_fraction = overlap_mm / 5.0
            for repeat in range(1, 6):
                rows.append({
                    "test_name": "footprintTest", "surface_value": 0,
                    "trial_num": trial_num, "sample_num": sample_num,
                    "sample_time": sample_num * 10_000 + repeat, "sensor_num": 2,
                    "sensor_reading": 250 + 1500 * dark_fraction + rng.normal(0, 10),
                    "rectangle_width_mm": rectangle_width_mm,
                    "direction": direction, "position_mm": position_mm,
                })
example_data = pd.DataFrame(rows)


## 3. Load and preview the selected data

This is where an uploaded CSV enters the notebook. Check that the column names, units and labels match the exercise before continuing.


In [ ]:
if USE_EXAMPLE_DATA:
    data = example_data.copy()
else:
    data = pd.read_csv(CSV_FILENAME)

data.head()


## 4. Enter measured references and analysis choices

Edit the reference table using new black and white measurements. Choose one sensor first. The threshold is operational and held-back positions must be selected before plotting.


In [ ]:
sensor_references = pd.DataFrame({
    "sensor_num": [0, 1, 2, 3, 4],
    "white_reference_us": [240, 245, 250, 255, 248],
    "black_reference_us": [1720, 1735, 1750, 1760, 1740],
})
TEST_NAME = "footprintTest"
SELECTED_SENSOR = 2
FOOTPRINT_THRESHOLD = 0.90
HELD_BACK_POSITIONS_MM = [-1.5, 1.5]
sensor_references


## 5. Plot every raw scan before estimating a footprint


In [ ]:
selected_raw = data.loc[
    (data["test_name"] == TEST_NAME) & (data["sensor_num"] == SELECTED_SENSOR)
].copy()
g = sns.relplot(
    data=selected_raw, x="position_mm", y="sensor_reading",
    hue="rectangle_width_mm", style="direction", kind="line",
    estimator="median", errorbar="sd", height=5, aspect=1.7,
)
g.set_axis_labels("Registered lateral position (mm)", "Raw discharge time (microseconds)")
g.figure.suptitle(f"Registered scans for sensor {SELECTED_SENSOR}", y=1.02)
plt.show()


## 6. Add a normalised response

`merge` attaches the selected sensor's independently measured references. Raw readings remain unchanged.


In [ ]:
analysis = selected_raw.merge(sensor_references, on="sensor_num", validate="many_to_one")
analysis["normalised_dark_response"] = (
    (analysis["sensor_reading"] - analysis["white_reference_us"])
    / (analysis["black_reference_us"] - analysis["white_reference_us"])
)
analysis["split"] = np.where(analysis["position_mm"].isin(HELD_BACK_POSITIONS_MM), "held back", "fit")
scan_summary = (
    analysis.groupby(["sensor_num", "rectangle_width_mm", "direction", "position_mm", "split"], as_index=False)
    .agg(response=("normalised_dark_response", "median"), response_sd=("normalised_dark_response", "std"))
)
scan_summary.head()


## 7. Estimate the footprint from peak and plateau evidence

The peak estimate assumes an idealised uniform one-dimensional footprint. The plateau estimate uses the editable threshold. Both are calculated separately for each traverse direction.


In [ ]:
fitting_summary = scan_summary.loc[scan_summary["split"] == "fit"]
estimate_rows = []
for (width_mm, direction), profile in fitting_summary.groupby(["rectangle_width_mm", "direction"]):
    profile = profile.sort_values("position_mm")
    maximum_response = profile["response"].max()
    high_response = profile.loc[profile["response"] >= FOOTPRINT_THRESHOLD]
    plateau_span_mm = 0.0
    if len(high_response) >= 2:
        plateau_span_mm = high_response["position_mm"].max() - high_response["position_mm"].min()
    estimate_rows.append({
        "rectangle_width_mm": width_mm, "direction": direction,
        "maximum_response": maximum_response, "plateau_span_mm": plateau_span_mm,
        "footprint_from_peak_mm": width_mm / maximum_response,
        "footprint_from_plateau_mm": width_mm - plateau_span_mm if plateau_span_mm > 0 else np.nan,
    })
direction_estimates = pd.DataFrame(estimate_rows)
direction_estimates


In [ ]:
footprint_summary = (
    direction_estimates.groupby("rectangle_width_mm", as_index=False)
    .agg(peak_estimate_mm=("footprint_from_peak_mm", "mean"),
         peak_direction_difference_mm=("footprint_from_peak_mm", "std"),
         plateau_estimate_mm=("footprint_from_plateau_mm", "mean"),
         plateau_direction_difference_mm=("footprint_from_plateau_mm", "std"))
)
footprint_summary


## 8. Predict positions kept out of profile construction

Interpolation uses neighbouring fitting positions from the same width and direction. It predicts a reserved reading; it does not verify the robot's physical position.


In [ ]:
held_back = scan_summary.loc[scan_summary["split"] == "held back"]
prediction_rows = []
for (width_mm, direction), test_rows in held_back.groupby(["rectangle_width_mm", "direction"]):
    training = fitting_summary.loc[
        (fitting_summary["rectangle_width_mm"] == width_mm)
        & (fitting_summary["direction"] == direction)
    ].sort_values("position_mm")
    predicted = test_rows.copy()
    predicted["predicted_response"] = np.interp(
        predicted["position_mm"], training["position_mm"], training["response"]
    )
    prediction_rows.append(predicted)
validation = pd.concat(prediction_rows, ignore_index=True)
validation["residual"] = validation["response"] - validation["predicted_response"]
validation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.scatterplot(data=validation, x="predicted_response", y="response",
                hue="rectangle_width_mm", style="direction", s=80, ax=axes[0])
limits = [min(validation["predicted_response"].min(), validation["response"].min()),
          max(validation["predicted_response"].max(), validation["response"].max())]
axes[0].plot(limits, limits, color="black", linestyle="--")
axes[0].set(title="Held-back measured versus predicted response")
sns.boxplot(data=validation, x="rectangle_width_mm", y="residual", hue="direction", ax=axes[1])
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(title="Held-back prediction residuals")
plt.tight_layout()
plt.show()


## What to notice

- Which widths produce a peak, a plateau, or neither?
- Do the two traverse directions give similar footprint estimates?
- How sensitive is the result to the chosen threshold?
- Repeat with each sensor's own references before comparing all five.
